Data-Loader

Imports

In [1]:
import pandas as pd
import numpy as np
import random 
from typing import List,Dict,Any,Tuple
import torch
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import time
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import torch
import os
import pickle
import numpy as np
import random
from tqdm import tqdm
import numpy as np
import random
import torch
import os



In [2]:
def preprocess_gat_raj_data(
        df_raw: pd.DataFrame,
        training: bool = True,
        global_scale: float = 10.0,
        displacement_scale: float = 5.0
) -> List[Dict[str, Any]]:
    """
    Preprocesses trajectory data for the GAT-TCN-Transformer model.

    Args:
        df_raw: Raw dataframe with UTM coordinates.
        training: Whether to apply random rotation.
        global_scale: Scale factor to divide global coordinates (meters -> scene units).
        displacement_scale: Scale for normalizing displacement (velocity).

    Returns:
        List of dictionaries, each representing a processed trajectory segment.
    """

    # ------------------------------------------------------------------
    # 0. GLOBAL UTM NORMALIZATION (CRITICAL)
    # ------------------------------------------------------------------
    # Step A: Convert UTM to local 0-based coordinates
    df_raw['x [m]'] = df_raw['x [m]'] - df_raw['x [m]'].min()
    df_raw['y [m]'] = df_raw['y [m]'] - df_raw['y [m]'].min()

    # Step B: Scale down large meter values (recommended: /10 or /20)
    df_raw['x [m]'] = df_raw['x [m]'] / global_scale
    df_raw['y [m]'] = df_raw['y [m]'] / global_scale

    # ------------------------------------------------------------------
    # 1. RENAME COLUMNS + SORT
    # ------------------------------------------------------------------
    needed_cols = ['Time', 'Track ID', 'x [m]', 'y [m]']
    df_cleaned = df_raw[needed_cols].copy()

    df_cleaned.rename(columns={
        'Time': 'Frame ID',
        'Track ID': 'Agent ID',
        'x [m]': 'x',
        'y [m]': 'y'
    }, inplace=True)

    df_cleaned.sort_values(by=['Frame ID', 'Agent ID'], inplace=True)

    # ------------------------------------------------------------------
    # 2. GROUP BY AGENT
    # ------------------------------------------------------------------
    groups = df_cleaned.groupby('Agent ID')

    obs_len = 8
    pred_len = 12
    tot_len = obs_len + pred_len

    all_segments = []

    # ------------------------------------------------------------------
    # 3. PROCESS EACH AGENT INTO TRAJECTORY SEGMENTS
    # ------------------------------------------------------------------
    for agent_id, agent_data in groups:
        frames = agent_data['Frame ID'].to_numpy()
        coords = agent_data[['x', 'y']].to_numpy()
        n = len(coords)

        if n < obs_len + 1:
            continue

        # Slide over the agent trajectory
        for i in range(n - obs_len):
            start = i
            end = i + obs_len
            true_end = min(i + tot_len, n)

            # Extract coordinates for this segment
            seg_abs = coords[start:true_end]

            # Pad future frames if needed
            if seg_abs.shape[0] < tot_len:
                pad = np.zeros((tot_len - seg_abs.shape[0], 2))
                seg_abs = np.vstack([seg_abs, pad])

            # ----------------------------------------------------------
            # 3A. SHIFT (RELATIVE POSITION)  → Stabilizes GAT adjacency
            # ----------------------------------------------------------
            P_obs = seg_abs[obs_len - 1]                      # anchor point
            P_shifted = seg_abs - P_obs                       # relative coords

            # ----------------------------------------------------------
            # 3B. DISPLACEMENT (VELOCITY)
            # ----------------------------------------------------------
            disp = P_shifted[1:] - P_shifted[:-1:]
            disp = np.vstack([np.zeros((1, 2)), disp])         # pad first frame

            # ----------------------------------------------------------
            # 3C. NORMALIZE DISPLACEMENT (IMPORTANT FOR TCN)
            # ----------------------------------------------------------
            disp_norm = disp / displacement_scale

            # ----------------------------------------------------------
            # 3D. DATA AUGMENTATION (RANDOM ROTATION)
            # ----------------------------------------------------------
            if training:
                theta = random.uniform(0, 2 * np.pi)
                c, s = np.cos(theta), np.sin(theta)
                R = np.array([[c, -s], [s, c]])

                P_shifted = P_shifted @ R.T
                disp_norm = disp_norm @ R.T

            # ----------------------------------------------------------
            # 3E. SAVE SEGMENT
            # ----------------------------------------------------------
            all_segments.append({
                'agent_id': agent_id,
                'start_frame': frames[start],
                'coords_abs': seg_abs.copy(),                      # scaled absolute
                'obs_coords_shifted': P_shifted[:obs_len],         # [8,2]
                'obs_displacement': disp_norm[:obs_len],           # [8,2] normalized
                'pred_displacement_gt': disp_norm[obs_len:tot_len],# [12,2] normed
                'shift_value': P_obs,                              # anchor for reconstruction
                'global_scale': global_scale,
                'displacement_scale': displacement_scale
            })

    return all_segments




class Data_Loader:
  def __init__(self,segments:List[Dict[str,Any]],args : Any):
    self.segments = segments
    self.args = args
    self.obs_len = args.obs_length
    self.pred_len = args.pred_length
    self.tot_len = self.obs_len+self.pred_len
    self.neighbour_threshold = args.neighbor_thred

    #collecting the frame from the segments

    self.segments_by_frame = self.group_by_frame(segments)
    #getting all the frame ids

    self.all_frame_ids = list(self.segments_by_frame.keys())

  def group_by_frame(self,segments:List[Dict[str,Any]]) -> Dict[int,List[Dict[str,Any]]]:
    frame_dict = {}
    for seg in segments:
      frame = seg['start_frame']
      if frame not in frame_dict:
        frame_dict[frame] = []
      frame_dict[frame].append(seg)
    return frame_dict

  def __len__(self) ->int :
    return len(self.all_frame_ids)

  def build_scene(self,scene_segments):
   t = self.tot_len
   n = len(scene_segments)

   abs_s = torch.zeros((t,n,2),dtype=torch.float32)
   norm_s = torch.zeros((t,n,2),dtype=torch.float32)
   nei_lists = torch.zeros((self.obs_len,n,n),dtype=torch.float32)

   for i, seg in enumerate(scene_segments):
    P_shifted = torch.from_numpy(seg['coords_abs']).float()
    abs_s[:,i,:] = P_shifted
    last_obs = abs_s[self.obs_len - 1, i, :].clone()  
    norm_s[:, i, :] = abs_s[:, i, :] - last_obs


   for i in range (self.obs_len):
     coords = abs_s[i]
     diff = coords[:,None] - coords[None,:]
     dist = torch.norm(diff,dim=2)
     spatial_adj = (dist < self.neighbour_threshold).float()
     spatial_adj.fill_diagonal_(0)
     mask_i = (coords.sum(dim=1) != 0).float()  
     spatial_adj = spatial_adj * mask_i[:, None] * mask_i[None, :]
     nei_lists[i] = spatial_adj
   return abs_s,norm_s,nei_lists


  def get_batch(self, frame_indices):
    """
    Builds a TRUE GATraj-style batch from multiple scenes.
    Input:
        frame_indices: list of frame indexes (batch_size scenes)
    Output:
        batch_abs_gt   [T, N_total, 2]
        batch_norm_gt  [T, N_total, 2]
        nei_list_batch list: each scene → [obs_len, n_s, n_s]
        nei_num_batch  [obs_len, N_total]
        batch_split    [[start_idx, end_idx], ...]
    """

    all_abs = []
    all_norm = []
    all_nei = []
    batch_split = []
    all_shifts =[]

    cur_start = 0

    for fi in frame_indices:
        frame_id = self.all_frame_ids[fi]
        scene = self.segments_by_frame[frame_id]

        # Build the scene (abs_s, norm_s, nei_s)
        abs_s, norm_s, nei_s = self.build_scene(scene)
        n_s = abs_s.shape[1]
        
        shift_s = torch.stack(
            [torch.from_numpy(seg['shift_value']).float() for seg in scene]
            ,dim=0
        )

        # Record scene boundaries
        batch_split.append([cur_start, cur_start + n_s])
        cur_start += n_s

        # Store scene tensors
        all_abs.append(abs_s)
        all_norm.append(norm_s)
        all_nei.append(nei_s)
        all_shifts.append(shift_s)

    # Merge scenes along agent dimension
    batch_abs_gt = torch.cat(all_abs, dim=1).float()   # [T, N_total, 2]
    batch_norm_gt = torch.cat(all_norm, dim=1).float() # [T, N_total, 2]
    shift_value_batch = torch.cat(all_shifts, dim=0).float() # [N_total, 2]
    # Build neighbor counts [obs_len, N_total]
    nei_num_batch = []
    for t in range(self.obs_len):
        per_timestep_counts = [torch.sum(nei[t], dim=1) for nei in all_nei]
        nei_num_batch.append(torch.cat(per_timestep_counts, dim=0))

    nei_num_batch = torch.stack(nei_num_batch, dim=0) 
    seq_list = (batch_abs_gt.sum(dim=2) != 0).float()
    
    nei_list_batch = all_nei
    return (batch_abs_gt, batch_norm_gt, nei_list_batch, nei_num_batch, seq_list,shift_value_batch,batch_split)

  def get_train_batch(self):
    """
    Randomly sample 'batch_size' scenes and return a merged GATraj-style batch.
    """
    idxs = np.random.choice(
        len(self.all_frame_ids),
        self.args.batch_size,
        replace=False
    )

    return self.get_batch(idxs)

  def get_val_batch(self):
    """
    Generator for validation batches.
    Iterates through the dataset in chunks of 'batch_size'.
    """
    idxs = np.arange(len(self.all_frame_ids))
    for i in range(0, len(idxs), self.args.batch_size):
        yield self.get_batch(idxs[i:i+self.args.batch_size])

  def get_test_batch(self):
    """
    Generator for test batches.
    Iterates through the dataset in chunks of 'batch_size'.
    """
    idxs = np.arange(len(self.all_frame_ids))
    for i in range(0, len(idxs), self.args.batch_size):
        yield self.get_batch(idxs[i:i+self.args.batch_size])
 




Models-GAT-TCN-Transformer

In [3]:
# ---------------------------
# Utility: Positional Encoding
# ---------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)  # [max_len, d_model]

    def forward(self, x):
        # x shape: [seq_len, batch, d_model] or [batch, seq_len, d_model]
        if x.dim() == 3 and x.shape[0] <= self.pe.shape[0]:
            # assume [seq_len, batch, d_model]
            seq_len = x.shape[0]
            return x + self.pe[:seq_len].unsqueeze(1).to(x.device)
        elif x.dim() == 3:
            # maybe [batch, seq_len, d_model]
            seq_len = x.shape[1]
            return x + self.pe[:seq_len].unsqueeze(0).to(x.device)
        else:
            return x

# ---------------------------
# Spatial GAT Implementation
# ---------------------------
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, concat=True, dropout=0.0, alpha=0.2):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.concat = concat
        self.W = nn.Linear(in_dim, out_dim, bias=False)       # Wh
        self.a = nn.Linear(2 * out_dim, 1, bias=False)        # attention vector
        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h, adj_mask):
        """
        h: [N, in_dim]
        adj_mask: [N, N] bool (1 neighbor, 0 not neighbor)
        returns: [N, out_dim]
        """
        device = h.device
        N = h.size(0)
        Wh = self.W(h)  # [N, out_dim]

        # prepare pairwise combinations efficiently:
        Wh_i = Wh.unsqueeze(1).expand(-1, N, -1)  # [N, N, out_dim]
        Wh_j = Wh.unsqueeze(0).expand(N, -1, -1)  # [N, N, out_dim]
        cat = torch.cat([Wh_i, Wh_j], dim=-1)     # [N, N, 2*out_dim]

        e = self.leakyrelu(self.a(cat).squeeze(-1))  # [N, N]

        # mask out non-neighbors
        if adj_mask is None:
            mask = torch.ones_like(e, dtype=torch.bool, device=device)
        else:
            mask = adj_mask.to(torch.bool)

        # ensure self-loop present
        diag_idx = torch.arange(0, N, device=device)
        mask[diag_idx, diag_idx] = True

        neg_inf = -9e15
        e_masked = e.masked_fill(~mask, neg_inf)

        alpha = torch.softmax(e_masked, dim=1)  # normalize over j for each i -> [N, N]
        alpha = self.dropout(alpha)

        h_prime = torch.matmul(alpha, Wh)  # [N, out_dim]

        if self.concat:
            return F.elu(h_prime)
        else:
            return h_prime

class MultiHeadGAT(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads=2, dropout=0.0, alpha=0.2, last_layer=False):
        super().__init__()
        assert out_dim % num_heads == 0, "out_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = out_dim // num_heads
        self.heads = nn.ModuleList([
            GATLayer(in_dim, self.head_dim, concat=(not last_layer), dropout=dropout, alpha=alpha)
            for _ in range(num_heads)
        ])
        self.last_layer = last_layer

    def forward(self, h, adj_mask):
        head_outs = [head(h, adj_mask) for head in self.heads]  # each [N, head_dim]
        # Always concatenate heads to get out_dim
        out = torch.cat(head_outs, dim=-1)  # [N, out_dim = num_heads * head_dim]
        return out

class SpatialGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_heads=2, dropout=0.0):
        super().__init__()
        # first layer: concat heads -> hidden_dim
        self.gat1 = MultiHeadGAT(in_dim, hidden_dim, num_heads=num_heads, dropout=dropout, last_layer=False)
        # second layer: average heads -> out_dim
        self.gat2 = MultiHeadGAT(hidden_dim, out_dim, num_heads=num_heads, dropout=dropout, last_layer=True)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, h, adj_mask):
        # h: [N, in_dim]
        x = self.gat1(h, adj_mask)  # [N, hidden_dim]
        x = self.gat2(x, adj_mask)  # [N, out_dim]
        x = self.norm(x)
        return x  # [N, out_dim]

# ---------------------------
# Temporal TCN (small, residual)
# ---------------------------
class TemporalTCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3, num_layers=2, dropout=0.0):
        super().__init__()
        layers = []
        for i in range(num_layers):
            in_ch = in_channels if i == 0 else hidden_channels
            dilation = 1
            padding = (kernel_size - 1) // 2  # keep sequence length same
            conv = nn.Conv1d(in_ch, hidden_channels, kernel_size, padding=padding, dilation=dilation)
            layers.append(nn.Sequential(conv, nn.ReLU(), nn.Dropout(dropout)))
        self.net = nn.Sequential(*layers)
        # final projection (optional)
        self.res_proj = nn.Conv1d(in_channels, hidden_channels, 1) if in_channels != hidden_channels else None

    def forward(self, x):
        """
        x: [batch=N, channels, seq_len]  (we treat N=agents as batch)
        returns: [batch=N, hidden_channels, seq_len]
        """
        res = self.res_proj(x) if self.res_proj is not None else x
        out = self.net(x)
        return out + res  # residual

# ---------------------------
# Transformer Decoder for trajectories
# ---------------------------
class TransformerTrajectoryDecoder(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=4, pred_len=12, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.pred_len = pred_len
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model*4, dropout=dropout, batch_first=False)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.pos_enc = PositionalEncoding(d_model, max_len=pred_len+50)
        # head to produce mean (mu) and scale (sigma)
        self.mu_head = nn.Linear(d_model, 2)
        self.sigma_head = nn.Linear(d_model, 2)
        self.min_scale = 1e-3

    def forward(self, memory):
        """
        memory: [mem_len, batch=N, d_model] (mem_len = obs_len)
        returns:
            loc: [pred_len, N, 2]
            scale: [pred_len, N, 2]  (positive)
        """
        device = memory.device
        N = memory.shape[1]
        # prepare tgt as zeros with positional enc
        tgt = torch.zeros(self.pred_len, N, self.d_model, device=device)  # [pred_len, N, d_model]
        tgt = self.pos_enc(tgt)
        mem = memory  # already assumed pos-encoded by caller if required
        out = self.transformer_decoder(tgt, mem)  # [pred_len, N, d_model]
        loc = self.mu_head(out)  # [pred_len, N, 2]
        sigma_raw = self.sigma_head(out)  # [pred_len, N, 2]
        sigma = F.softplus(sigma_raw) + self.min_scale
        return loc, sigma

# ---------------------------
# Gaussian NLL Loss (bivariate independent dims)
# ---------------------------
class GaussianNLLLoss(nn.Module):
    def __init__(self, eps=1e-6, reduction='mean'):
        super().__init__()
        self.eps = eps
        self.reduction = reduction

    def forward(self, pred, target):
        """
        pred: concatenated loc and scale OR a tuple (loc, scale)
        - loc: [pred_len, N, 2]
        - scale: [pred_len, N, 2] (positive)
        target: [pred_len, N, 2]
        return: scalar loss
        """
        if isinstance(pred, tuple) or isinstance(pred, list):
            loc, scale = pred
        else:
            # assume last dim concatenation
            raise ValueError("pred should be (loc, scale) tuple")
        # compute elementwise nll assuming independent dims:
        # nll = 0.5 * (log(2*pi) + 2*log(scale) + ((y - loc)^2) / scale^2)
        scale = scale.clone()
        scale = scale.clamp_min(self.eps)
        diff2 = (target - loc) ** 2
        nll = 0.5 * (torch.log(2 * math.pi * (scale ** 2)) + diff2 / (scale ** 2))
        if self.reduction == 'mean':
            return nll.mean()
        elif self.reduction == 'sum':
            return nll.sum()
        else:
            return nll

# ---------------------------
# Top-level model wrapper
# ---------------------------
class GAT_TCN_Transformer(nn.Module):
    """
    Wrapper that replicates the old GATraj forward signature:
    forward(inputs, epoch, iftest=False)
    where inputs = (batch_abs_gt, batch_norm_gt, nei_list_batch, nei_num_batch, batch_split)
    """
    def __init__(self, args):
        super().__init__()
        self.args = args
        hidden = getattr(args, "hidden_size", 64)
        self.hidden_size = hidden
        self.obs_len = args.obs_length
        self.pred_len = args.pred_length

        # Spatial GAT: input dim is 2 (x,y) unless you change to include velocities
        self.spat_gat = SpatialGAT(in_dim=2, hidden_dim=hidden, out_dim=hidden, num_heads=2, dropout=0.0)

        # TCN: we use agent as batch dimension, channels = hidden, seq_len = obs_len
        self.tcn = TemporalTCN(in_channels=hidden, hidden_channels=hidden, kernel_size=getattr(args, "tcn_kernel", 3),
                               num_layers=getattr(args, "tcn_layers", 2), dropout=getattr(args, "tcn_dropout", 0.0))

        # a light linear to map tcn features to transformer d_model (hidden)
        self.memory_proj = nn.Linear(hidden, hidden)

        # positional encoding for memory (obs frames)
        self.mem_pos_enc = PositionalEncoding(hidden, max_len=self.obs_len+50)

        # Transformer decoder
        self.decoder = TransformerTrajectoryDecoder(d_model=hidden, nhead=getattr(args, "transformer_heads", 4),
                                                    num_layers=getattr(args, "transformer_layers", 4),
                                                    pred_len=self.pred_len, dropout=0.1)

        # loss
        self.reg_loss = GaussianNLLLoss(reduction='mean')

    # helper to build a full adjacency mask across the concatenated batch
    def build_adj_mask_for_frame(self, nei_list_batch, batch_split, t, device):
        """
        nei_list_batch: list length = num_scenes_in_minibatch
           each element: [H, N_scene, N_scene] (numpy or list)
        batch_split: list of [left, right] pairs indicating index range in full batch
        t: time index (0..obs_len-1)
        returns: [N_total, N_total] bool mask
        """
        # compute total agents:
        total_agents = 0
        for (l, r) in batch_split:
            total_agents += (r - l)
        adj = torch.zeros((total_agents, total_agents), dtype=torch.bool, device=device)
        # fill blocks
        for b_idx, (lr) in enumerate(batch_split):
            left, right = lr[0], lr[1] if isinstance(lr, (list, tuple)) else (lr[0], lr[1])
            # in original code batch_split elements are [start, end]
            left, right = lr[0], lr[1] if isinstance(lr, (list, tuple)) else (lr[0], lr[1])
        # Above handling is defensive; simpler:
        for b_idx, br in enumerate(batch_split):
            left, right = br[0], br[1]
            # nei_list_batch[b_idx] is either a list or numpy array: shape [H, N_scene, N_scene]
            nei_scene = nei_list_batch[b_idx]
            # ensure it's tensor
            if isinstance(nei_scene, list):
                nei_scene = torch.tensor(nei_scene, device=device)
            else:
                nei_scene = torch.tensor(nei_scene, device=device) if not torch.is_tensor(nei_scene) else nei_scene.to(device)
            # some datasets store nei_list for full seq_length, with indices aligned; pick frame t
            # clip t to available length if needed
            if t >= nei_scene.shape[0]:
                t_idx = nei_scene.shape[0] - 1
            else:
                t_idx = t
            block = nei_scene[t_idx].bool()  # [N_scene, N_scene]
            adj[left:right, left:right] = block
        # ensure diagonal self loops
        idx = torch.arange(0, adj.shape[0], device=device)
        adj[idx, idx] = True
        return adj

    def forward(self, inputs, epoch=None, iftest=False):
        """
        inputs: tuple:
          batch_abs_gt: [H, N, 2]
          batch_norm_gt: [H, N, 2]
          nei_list_batch: list (per scene) each [H, N_scene, N_scene]
          nei_num_batch: [N, H] or similar (unused here)
          batch_split: list of [left, right] ranges
        returns:
          loss, [best_prediction_trajectories_list]
        """
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        batch_abs_gt, batch_norm_gt, nei_list_batch, nei_num_batch, batch_split = inputs
        # Ensure tensors
        batch_norm_gt = batch_norm_gt.to(device)
        # observed frames positions
        obs_pos = batch_norm_gt[:self.obs_len, :, :2]  # [obs_len, N, 2]
        H, N, _ = obs_pos.shape

        # collect spatial embeddings per frame
        spatial_emb_list = []
        for t in range(self.obs_len):
            # node features: [N, 2]
            h_t = obs_pos[t].to(device)  # [N, 2]
            # build adjacency mask for frame t
            adj_mask_t = self.build_adj_mask_for_frame(nei_list_batch, batch_split, t, device)  # [N, N]
            # Spatial GAT expects float features; adj boolean mask
            gat_out = self.spat_gat(h_t.float(), adj_mask_t)  # [N, hidden]
            spatial_emb_list.append(gat_out.unsqueeze(0))  # [1, N, hidden]

        # stack: [obs_len, N, hidden]
        spatial_emb = torch.cat(spatial_emb_list, dim=0)
        # transform to [N, hidden, obs_len] for TCN (batch=agents)
        spatial_emb_batch = spatial_emb.permute(1, 2, 0)  # [N, hidden, obs_len]

        # run TCN (agent-as-batch)
        tcn_out = self.tcn(spatial_emb_batch)  # [N, hidden, obs_len]
        # project hidden -> d_model (same here)
        tcn_out = tcn_out.permute(2, 0, 1)  # [obs_len, N, hidden]
        # optional positional encoding on memory
        tcn_out = self.mem_pos_enc(tcn_out)  # [obs_len, N, hidden]

        # decode with Transformer decoder
        loc, scale = self.decoder(tcn_out)  # both [pred_len, N, 2]

        # compute loss against ground truth normalized positions
        # predicted y should be compared to batch_norm_gt[self.obs_len:, :, :2] shape [pred_len, N, 2]
        target = batch_norm_gt[self.obs_len:, :, :2].to(device)  # [pred_len, N, 2]
        # ensure shapes match
        if target.shape[0] != loc.shape[0]:
            # handle mismatch: crop/pad target or loc
            L = min(target.shape[0], loc.shape[0])
            target = target[:L]
            loc = loc[:L]
            scale = scale[:L]

        loss = self.reg_loss((loc, scale), target)

        # Build full predicted trajectory(s) for evaluation
        # The evaluation expects: obs frames [1:obs_len] + pred frames = total 19 frames
        # obs frames 1 to 7 (7 frames) + pred frames 8-19 (12 frames) = 19 frames
        pre_obs = batch_norm_gt[1:self.obs_len, :, :2].to(device)  # [obs_len-1, N, 2] = [7, N, 2]
        pred_cumsum = torch.cumsum(loc, dim=0)  # [pred_len, N, 2] = [12, N, 2]
        
        full_pre_tra = []
        # Concatenate observed trajectory (from frame 1) with predictions
        full_traj = torch.cat([pre_obs, pred_cumsum], dim=0)  # [7+12, N, 2] = [19, N, 2]
        full_pre_tra.append(full_traj)

        return loss, full_pre_tra

# For backward compatibility naming:
GAT_TCN_Transformer = GAT_TCN_Transformer


Processor.py

In [4]:
class Processor():
    def __init__(self, args):
        self.args = args
        csv_path = os.path.join(self.args.base_dir,self.args.csv_data_path)
        df_raw = pd.read_csv(csv_path)
        training_mode=(self.args.phase=="train")
        segments=preprocess_gat_raj_data(df_raw,training=training_mode)
        self.lr=self.args.learning_rate
        self.dataloader_gt = Data_Loader(segments,args)
        self.net = GAT_TCN_Transformer(args)
        if self.args.phase == "train":
            print("self.args.phase",self.args.phase)
            self.net.train()
        else:
            self.net.eval()
        self.init_lr = self.args.learning_rate
        self.step_ratio = self.args.step_ratio
        self.lr_step=self.args.lr_step
        self.set_optimizer()
        self.epoch = 0
        self.load_model()
        # self.save_model(self.epoch)
        if self.args.using_cuda:
            self.net=self.net.cuda()
        else:
            self.net=self.net.cpu()
        self.net_file = open(os.path.join(self.args.model_dir, 'net.txt'), 'a+')
        self.net_file.write(str(self.net))
        self.net_file.close()
        self.log_file_curve = open(os.path.join(self.args.model_dir, 'log_curve.txt'), 'a+')



    def save_model(self,epoch):
        model_path= self.args.save_dir + '/' + self.args.train_model + '/' + self.args.train_model + '_' +\
                                   str(epoch) + '.tar'
        torch.save({
            'epoch': epoch,
            'state_dict': self.net.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict()
        }, model_path,_use_new_zipfile_serialization=False)


    def load_model(self):
        if self.args.load_model > 0:
            self.args.model_save_path = self.args.save_dir + '/' + self.args.train_model + '/' + self.args.train_model + '_' + \
                                        str(self.args.load_model) + '.tar'
            if os.path.isfile(self.args.model_save_path):
                print('Loading checkpoint')
                checkpoint = torch.load(self.args.model_save_path,map_location={'cuda:0': 'cuda:'+str(self.args.gpu)})
                # print("self.args.model_save_path",self.args.model_save_path)
                model_epoch = checkpoint['epoch']
                self.epoch = int(model_epoch) + 1
                self.net.load_state_dict(checkpoint['state_dict'])
                print('Loaded checkpoint at epoch', model_epoch)
                for i in range(self.args.load_model):
                    self.scheduler.step()


    def set_optimizer(self):
        self.optimizer = torch.optim.Adam(self.net.parameters(),lr=self.lr)
        self.criterion = nn.MSELoss(reduction='none')
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=self.optimizer,\
        T_max = self.args.num_epochs, eta_min=self.args.eta_min)

    def playtest(self):
        print('Testing begin')
        test_error, test_final_error, _= self.test_epoch(self.args.load_model)
        print('Set: {}, epoch: {:.5f},test_error: {:.5f} test_final_error: {:.5f}'.format(self.args.test_set,self.args.load_model,test_error,test_final_error))

    def playtrain(self):
        print('Training begin')
        test_error, test_final_error,first_erro_test,val_final_error,val_error,val_erro_first=0,0,0,0,0,0
        for epoch in range(self.epoch, self.args.num_epochs+1):
            print('Epoch-{0} lr: {1}'.format(epoch, self.optimizer.param_groups[0]['lr']))
            train_loss = self.train_epoch(epoch)
            val_error, val_final_error, val_erro_first = self.val_epoch(epoch)
            self.scheduler.step()
            # if epoch == self.args.num_epochs:
            self.save_model(epoch)
            if epoch == self.args.num_epochs:
                test_error, test_final_error, first_erro_test = self.test_epoch(epoch)
            #log files
            self.log_file_curve.write(str(epoch) + ',' + str(train_loss) + ',' + str(
                val_error) + ',' + str(val_final_error) + ','+str(val_erro_first)+ ','\
                +str(test_error) + ',' + str(test_final_error) + ','+str(first_erro_test)+ '\n')

            self.log_file_curve.close()
            self.log_file_curve = open(os.path.join(self.args.model_dir, 'log_curve.txt'), 'a+')
            #console log
            print('----epoch {} \n train_loss={:.5f}, valid_error={:.3f}, valid_final={:.3f}, valid_first={:.3f}\n\
                test_error={:.3f},test_final={:.3f},test_first={:.3f}\n'\
            .format(epoch, train_loss,val_error, val_final_error,val_erro_first,test_error,test_final_error,first_erro_test))
            model_path= self.args.save_dir + '/' + self.args.train_model + '/' + self.args.train_model + '_' + str(epoch) + '.tar'



    def train_epoch(self,epoch):
        """   batch_abs: the (orientated) batch
              batch_norm: the batch shifted by substracted the last position. ??? What is the impact of zeros
              shift_value: the last observed position
              seq_list: [seq_length, num_peds], mask for position with actual values at each frame for each ped
              nei_list: [seq_length, num_peds, num_peds], mask for neigbors at each frame
              nei_num: [seq_length, num_peds], neighbors at each frame for each ped
              batch_pednum: list, number of peds in each batch"""
        self.net.train()
        loss_epoch=0
        num_batches = len(self.dataloader_gt)//self.args.batch_size
        for batch in range(num_batches):
            start = time.time()
            batch_abs_gt,batch_norm_gt,nei_lists,nei_num,seq_list_gt,shift_value_gt,batch_split = self.dataloader_gt.get_train_batch()
            if self.args.using_cuda:
                batch_abs_gt=batch_abs_gt.cuda()
                batch_norm_gt=batch_norm_gt.cuda()
                nei_lists=[nei.cuda() for nei in nei_lists]
                nei_num=nei_num.cuda()
                seq_list_gt=seq_list_gt.cuda()
                shift_value_gt=shift_value_gt.cuda()

            inputs_fw=(batch_abs_gt,
                       batch_norm_gt,
                       nei_lists,
                       nei_num,
                       batch_split)

            self.net.zero_grad()

            GATraj_loss, full_pre_tra = self.net.forward(inputs_fw, epoch, iftest=False)
            if GATraj_loss == 0:
                continue
            loss_epoch += GATraj_loss.item()
            GATraj_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.net.parameters(), self.args.clip)
            self.optimizer.step()
            end= time.time()
                    # Logging
            if batch % self.args.show_step == 0 and self.args.ifshow_detail:
              print(
                f"train-{batch}/{num_batches} (epoch {epoch}), "
                f"train_loss = {GATraj_loss.item():.5f}, time/batch = {end-start:.5f}"
              )

        train_loss_epoch = loss_epoch / num_batches

        return train_loss_epoch

    def val_epoch(self, epoch):
        self.net.eval()
        error_epoch, final_error_epoch, first_erro_epoch = 0, 0, 0
        error_cnt_epoch, final_error_cnt_epoch, first_erro_cnt_epoch = 1e-5, 1e-5, 1e-5

        batch_count = 0
        num_val_batches = len(self.dataloader_gt) // self.args.batch_size

        for batch_data in self.dataloader_gt.get_val_batch():

            if batch_count % 100 == 0:
                print(f"validating batch {batch_count}/{num_val_batches}")

            # Custom loader returns 7 values
            batch_abs_gt, batch_norm_gt, nei_lists, nei_num, seq_list_gt, shift_value_gt, batch_split = batch_data

            # Move to GPU
            if self.args.using_cuda:
                batch_abs_gt = batch_abs_gt.cuda()
                batch_norm_gt = batch_norm_gt.cuda()
                nei_lists=[nei.cuda() for nei in nei_lists]
                nei_num = nei_num.cuda()
                seq_list_gt = seq_list_gt.cuda()
                shift_value_gt = shift_value_gt.cuda()

            # Model forward
            inputs_fw = (batch_abs_gt, batch_norm_gt, nei_lists, nei_num, batch_split)
            GATraj_loss, full_pre_tra = self.net.forward(inputs_fw, epoch, iftest=True)

            if GATraj_loss == 0:
                continue

            # Compute errors for each predicted mode
            error_list = []
            first_list = []
            final_list = []

            for pre_tra in full_pre_tra:
                error, error_cnt, final_error, final_error_cnt, first_erro, first_erro_cnt = \
                    L2forTest(pre_tra, batch_norm_gt[1:, :, :2], self.args.obs_length)

                error_list.append(error)
                first_list.append(first_erro)
                final_list.append(final_error)

            # Use minimum error mode
            error_epoch += min(error_list)
            final_error_epoch += min(final_list)
            first_erro_epoch += min(first_list)

            error_cnt_epoch += error_cnt
            final_error_cnt_epoch += final_error_cnt
            first_erro_cnt_epoch += first_erro_cnt

            batch_count += 1

        return (
            error_epoch / error_cnt_epoch,
            final_error_epoch / final_error_cnt_epoch,
            first_erro_epoch / first_erro_cnt_epoch
        )


    def test_epoch(self, epoch):
        self.net.eval()
        error_epoch, final_error_epoch, first_erro_epoch = 0, 0, 0
        error_cnt_epoch, final_error_cnt_epoch, first_erro_cnt_epoch = 1e-5, 1e-5, 1e-5

        batch_count = 0
        num_test_batches = len(self.dataloader_gt) // self.args.batch_size

        for batch_data in self.dataloader_gt.get_test_batch():

            if batch_count % 100 == 0:
                print(f"testing batch {batch_count}/{num_test_batches}")

            batch_abs_gt, batch_norm_gt, nei_lists, nei_num, seq_list_gt, shift_value_gt, batch_split = batch_data

            # Move to GPU
            if self.args.using_cuda:
                batch_abs_gt = batch_abs_gt.cuda()
                batch_norm_gt = batch_norm_gt.cuda()
                nei_lists=[nei.cuda() for nei in nei_lists]
                nei_num = nei_num.cuda()
                seq_list_gt = seq_list_gt.cuda()
                shift_value_gt = shift_value_gt.cuda()

            inputs_fw = (batch_abs_gt, batch_norm_gt, nei_lists, nei_num, batch_split)

            GATraj_loss, full_pre_tra = self.net.forward(inputs_fw, epoch, iftest=True)
            if GATraj_loss == 0:
                continue

            error_list = []
            first_list = []
            final_list = []

            for pre_tra in full_pre_tra:
                error, error_cnt, final_error, final_error_cnt, first_erro, first_erro_cnt = \
                    L2forTest(pre_tra, batch_norm_gt[1:, :, :2], self.args.obs_length)

                error_list.append(error)
                final_list.append(final_error)
                first_list.append(first_erro)

            error_epoch += min(error_list)
            final_error_epoch += min(final_list)
            first_erro_epoch += min(first_list)

            error_cnt_epoch += error_cnt
            final_error_cnt_epoch += final_error_cnt
            first_erro_cnt_epoch += first_erro_cnt

            batch_count += 1

        return (
            error_epoch / error_cnt_epoch,
            final_error_epoch / final_error_cnt_epoch,
            first_erro_epoch / first_erro_cnt_epoch
        )



In [5]:
def L2forTest(outputs,targets,obs_length):
    '''
    Evaluation.
    information: [N, 3]
    '''
    seq_length = outputs.shape[0]
    error = torch.norm(outputs-targets,p=2,dim=2)
    error_pred_length = error[obs_length-1:]
    error = torch.sum(error_pred_length)
    error_cnt = error_pred_length.numel()
    if error == 0:
        return 0,0,0,0,0,0
    final_error = torch.sum(error_pred_length[-1])
    final_error_cnt = error_pred_length[-1].numel()
    first_erro = torch.sum(error_pred_length[0])
    first_erro_cnt = error_pred_length[0].numel()
    return error.item(),error_cnt,final_error.item(),final_error_cnt,first_erro.item(),first_erro_cnt

    
def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod

In [8]:
# Configuration class - Easy to modify
class Config:
    def __init__(self):
        # Training parameters
        self.phase = 'train'  # 'train' or 'test'
        self.num_epochs = 100
        self.batch_size = 8
        self.learning_rate = 1e-04
        self.clip = 10
        
        # Model architecture
        self.hidden_size = 64
        self.obs_length = 8
        self.pred_length = 12
        self.seq_length = 20
        
        # TCN parameters
        self.tcn_layers = 3
        self.tcn_kernel = 3
        self.tcn_dropout = 0.0
        
        # Transformer parameters
        self.transformer_heads = 4
        self.transformer_layers = 4
        
        # Data parameters
        self.neighbor_thred = 1.0
        self.randomRotate = True
        
        # Paths (Change these for your Colab setup)
        self.base_dir = './'
        self.csv_data_path = 'final_surajpur_proper_reduced_2000.csv'
        self.save_base_dir = './savedata/'
        
        # Other settings
        self.test_set = 1
        self.train_model = 'GATraj'
        self.load_model = 0  # Set > 0 to load checkpoint
        self.using_cuda = True
        self.gpu = 0
        self.show_step = 40
        self.ifshow_detail = True
        
        # Computed paths
        self.save_dir = self.save_base_dir + str(self.test_set) + '/'
        self.model_dir = self.save_dir + self.train_model + '/'
        self.config = self.model_dir + 'config_' + self.phase + '.yaml'
        
        # Optimizer parameters
        self.eta_min = 3e-6
        self.T_max = 1000
        self.lr_step = 20
        self.step_ratio = 0.5
        
        # Legacy compatibility params
        self.ifvalid = True
        self.num_pred = 1
        self.ratio = 0.95
        self.z_dim = 32
        self.x_encoder_layers = 3
        self.x_encoder_head = 8
        self.output_size = 2
        self.input_size = 2
        self.min_obs = 8

def prepare_seed(rand_seed):
    np.random.seed(rand_seed)
    random.seed(rand_seed)
    torch.manual_seed(rand_seed)
    torch.cuda.manual_seed_all(rand_seed)

# MAIN EXECUTION
args = Config()
prepare_seed(1)

if args.using_cuda:
    torch.cuda.set_device(args.gpu)

processor = Processor(args)

if args.phase == 'test':
    processor.playtest()
else:
    processor.playtrain()

self.args.phase train
Training begin
Epoch-0 lr: 0.0001
train-0/4 (epoch 0), train_loss = 17.43504, time/batch = 0.25386
validating batch 0/4
----epoch 0 
 train_loss=12.89505, valid_error=8.569, valid_final=15.521, valid_first=1.200
                test_error=0.000,test_final=0.000,test_first=0.000

Epoch-1 lr: 9.997606817773798e-05
train-0/4 (epoch 1), train_loss = 5.68828, time/batch = 0.27421
validating batch 0/4
----epoch 1 
 train_loss=5.21861, valid_error=8.837, valid_final=15.953, valid_first=1.288
                test_error=0.000,test_final=0.000,test_first=0.000

Epoch-2 lr: 9.990429632877117e-05
train-0/4 (epoch 2), train_loss = 4.15207, time/batch = 0.22014
validating batch 0/4
----epoch 2 
 train_loss=5.93098, valid_error=9.137, valid_final=16.423, valid_first=1.382
                test_error=0.000,test_final=0.000,test_first=0.000

Epoch-3 lr: 9.978475528324939e-05
train-0/4 (epoch 3), train_loss = 2.98105, time/batch = 0.15763
validating batch 0/4
----epoch 3 
 train_los